In [ ]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
import scipy
import mantid
from mantid.simpleapi import *
import scippneutron
import h5py
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import ipyvolume as ipv


In [ ]:
f_nexus = r"./magic/run_folder/mccode.h5" 

In [ ]:
Load(f_nexus, OutputWorkspace='mcstas_data')

In [ ]:
print(mtd.getObjectNames())

In [ ]:
from mantid.simpleapi import CreateWorkspace
ws = mtd['tof_egs2_1.plot_mcstas_data']

monitor = ws.readY(0)
mon_ws = CreateWorkspace(
    DataX=ws.readX(0),
    DataY=monitor,
    NSpec=1,
    UnitX=ws.getAxis(0).getUnit().unitID()
)

from mantid.simpleapi import Divide


In [ ]:
mtd['EventData_mcstas_data']

In [ ]:
mtd['EventData_mcstas_data']

In [ ]:
mtd['EventData_mcstas_data'].getNumberHistograms()

In [ ]:
mtd['EventData_mcstas_data'].getNumberEvents()

In [ ]:
from mantid.simpleapi import Rebin

# Convert to MD and visualize in Q‑space (if needed)


In [ ]:
from mantid.simpleapi import ConvertToMD

ConvertToDiffractionMDWorkspace(InputWorkspace='EventData_mcstas_data', OutputWorkspace='md_space')
md = mtd['md_space']

In [ ]:
md

In [ ]:
ws = md

dims = [ws.getDimension(i) for i in range(ws.getNumDims())]
xmin, xmax = dims[0].getMinimum(), dims[0].getMaximum()
ymin, ymax = dims[1].getMinimum(), dims[1].getMaximum()
zmin, zmax = dims[2].getMinimum(), dims[2].getMaximum()

extent_x = xmax - xmin
extent_y = ymax - ymin
extent_z = zmax - zmin

# total number of events
nevents = ws.getNPoints()

## Choose an automatic binning rule

For 3D event clouds, a good rule is:
Bins per axis ≈ (N_events)^(1/3) / k
where k controls smoothness:

 - k = 3 → fine resolution
 - k = 4–5 → medium
 - k = 6–8 → coarse

This is analogous to Scott’s rule / Freedman–Diaconis but adapted to 3D.

In [ ]:
k = 3  # smooth but not too coarse
nbins = int((nevents ** (1/3)) / k)

# enforce reasonable limits
nbins = max(20, min(nbins, 200))
nbins

In [ ]:
from mantid.simpleapi import BinMD

binned = BinMD(
    InputWorkspace=ws,
    AlignedDim0=f"Q_lab_x,{xmin},{xmax},{nbins}",
    AlignedDim1=f"Q_lab_y,{ymin},{ymax},{nbins}",
    AlignedDim2=f"Q_lab_z,{zmin},{zmax},{nbins}"
)

In [ ]:

data = binned.getSignalArray()  # shape (nx, ny, nz)

dims = [binned.getDimension(i) for i in range(3)]
qx = np.linspace(dims[0].getMinimum(), dims[0].getMaximum(), dims[0].getNBins())
qy = np.linspace(dims[1].getMinimum(), dims[1].getMaximum(), dims[1].getNBins())
qz = np.linspace(dims[2].getMinimum(), dims[2].getMaximum(), dims[2].getNBins())

QX, QY, QZ = np.meshgrid(qx, qy, qz, indexing="ij")


In [ ]:
import ipyvolume as ipv
import ipywidgets as widgets
import numpy as np
import matplotlib.cm as cm 
import matplotlib
import matplotlib.colors as colors

# Flatten arrays
qx = QX.flatten()
qy = QY.flatten()
qz = QZ.flatten()
val = data.flatten()

# Normalize for colormap
norm = colors.Normalize(vmin=val.min(), vmax=val.max())
cmap = matplotlib.colormaps.get_cmap("viridis")

# Initial threshold
initial_thr = np.percentile(val, 90)
mask = val > initial_thr

# Initial colors
col = cmap(norm(val[mask]))[:, :3]

# Create figure
fig = ipv.figure()
fig.camera.projection = 'orthographic'
scatter = ipv.scatter(
    qx[mask], qy[mask], qz[mask],
    color=col,
    size=3,
    marker="sphere"
)

ipv.show()

# --- Widgets ---

thr_slider = widgets.FloatSlider(
    value=initial_thr,
    min=float(val.min()),
    max=float(val.max()),
    step=(val.max() - val.min()) / 200,
    description='Threshold',
    continuous_update=False
)

proj_toggle = widgets.ToggleButton(
    value=False,
    description='Orthographic OFF',
    icon='cube'
)

# --- Callbacks ---

def update_threshold(change):
    thr = change["new"]
    mask = val > thr

    scatter.x = qx[mask]
    scatter.y = qy[mask]
    scatter.z = qz[mask]

    scatter.color = cmap(norm(val[mask]))[:, :3]

thr_slider.observe(update_threshold, names="value")

def update_projection(change):
    if change["new"]:
        fig.camera.projection = 'orthographic'
        proj_toggle.description = "Orthographic ON"
        proj_toggle.icon = "square"
    else:
        fig.camera.projection = 'perspective'
        proj_toggle.description = "Orthographic OFF"
        proj_toggle.icon = "cube"

proj_toggle.observe(update_projection, names="value")

widgets.VBox([thr_slider, proj_toggle])


# Find peaks in MD space

In [ ]:
FindPeaksMD(
    InputWorkspace="md_space", 
    OutputWorkspace="strong_peaks",
    PeakDistanceThreshold='0.1',
    MaxPeaks='100')

In [ ]:
def print_table(work_space_name:str, l_name_print=[]):
    work_space = mtd[work_space_name]
    l_column_name = work_space.getColumnNames()
    n_line = work_space.rowCount()
    ls_out = []
    ls_out.append(' '.join(['{:<10}'.format(str(hh)) for hh in l_column_name  if not(len(l_name_print)!=0 and not(hh in l_name_print)) ]))
    for i in range(n_line):
        l_hh = []
        for name in l_column_name:
            if len(l_name_print)!=0 and not(name in l_name_print):
                continue
            val = work_space.column(name)[i]
            if isinstance(val, float):
                sval = f"{val:10.3f}"
            elif isinstance(val, int):
                sval = f"{val:10}"
            else:
                sval = '{:<10}'.format(str(val))
            l_hh.append(sval)
        ls_out.append(' '.join(l_hh))
    return '\n'.join(ls_out)

In [ ]:
print(print_table("strong_peaks",l_name_print=['QLab', 'Wavelength']))

In [ ]:
np_qlab = numpy.array(mtd["strong_peaks"].column('QLab'), dtype=float).transpose()


In [ ]:
plt.scatter(np_qlab[0], np_qlab[1], c=np_qlab[2], alpha=0.75, s=50, cmap='Accent')
cbar = plt.colorbar()
cbar.set_label(r'$Q_z  (\AA^{-1})$')
plt.xlabel(r'$Q_x (\AA^{-1})$')
plt.ylabel(r'$Q_y (\AA^{-1})$')
plt.title('Found peaks in Q-space')

In [ ]:
plt.scatter(np_qlab[0], np_qlab[2], c=np_qlab[2], alpha=0.75, s=50, cmap='Accent')
cbar = plt.colorbar()
cbar.set_label(r'$Q_z  (\AA^{-1})$')
plt.xlabel(r'$Q_x (\AA^{-1})$')
plt.ylabel(r'$Q_z (\AA^{-1})$')
plt.title('Found peaks in Q-space')

In [ ]:
a, b, c = 14.04078, 14.04078, 14.04078
alpha, beta, gamma = 90, 90, 90
cell_type = 'Cubic'
centering = 'P'

CreateSingleValuedWorkspace(OutputWorkspace='sample')
SetUB(Workspace='sample', a=a, b=b, c=c, alpha=alpha, beta=beta, gamma=gamma)

# Searching UB-matrix

In [ ]:
tol = 0.3
FindUBUsingLatticeParameters(PeaksWorkspace="strong_peaks", a=a,b=b,c=c,alpha=alpha,beta=beta, gamma=gamma,Tolerance=tol)

In [ ]:
IndexPeaks(PeaksWorkspace="strong_peaks")


In [ ]:
print(print_table("strong_peaks",l_name_print=['h', 'k','l','Wavelength']))

In [ ]:
IntegratePeaksMD(
    InputWorkspace='md_space', 
    PeaksWorkspace='strong_peaks', 
    OutputWorkspace="integrated_strong_peaks",
    PeakRadius=0.049, 
    BackgroundOuterRadius=0.049, 
    BackgroundInnerRadius=0.098,
    IntegrateIfOnEdge=False
    )


In [ ]:
print(print_table("integrated_strong_peaks",l_name_print=['h', 'k', 'l', 'Intens', 'SigInt']))

In [ ]:
# SetUB(Workspace='md_space', a=a, b=b, c=c, alpha=alpha, beta=beta, gamma=gamma, UB= [1/a, 0, 0, 0, 1/b, 0, 0, 0, 1/c])
# SetUB(Workspace='strong_peaks', a=a, b=b, c=c, alpha=alpha, beta=beta, gamma=gamma, UB= [1/a, 0, 0, 0, 1/b, 0, 0, 0, 1/c])
# IndexPeaks(PeaksWorkspace="strong_peaks")


In [ ]:
np_hkl = numpy.stack([
    numpy.array(mtd["strong_peaks"].column('h'), dtype=float),
    numpy.array(mtd["strong_peaks"].column('k'), dtype=float),
    numpy.array(mtd["strong_peaks"].column('l'), dtype=float)], axis=0)
print(np_hkl)
plt.scatter(np_hkl[0], np_hkl[1], c=np_hkl[2], alpha=0.75, s=50, cmap='Accent')
cbar = plt.colorbar()
cbar.set_label(r'$l$')
plt.xlabel(r'$h$')
plt.ylabel(r'$k$')
plt.title('Found peaks in HKL-space')

In [ ]:
plt.scatter(np_hkl[0], np_hkl[2], c=np_hkl[2], alpha=0.75, s=50, cmap='Accent')
cbar = plt.colorbar()
cbar.set_label(r'$l$')
plt.xlabel(r'$h$')
plt.ylabel(r'$l$')
plt.title('Found peaks in HKL-space')

# Normalization per monitor

In [ ]:
ws_norm


In [ ]:
ws_tof

In [ ]:
from mantid.simpleapi import ConvertUnits
ws_tof = mtd['tof_egs2_1.plot_mcstas_data']
ws_tof.getAxis(0).setUnit("TOF")
ws_lambda = ConvertUnits(
    InputWorkspace=ws_tof,
    Target="Wavelength"
)
ws_lambda

In [ ]:
import numpy as np
from mantid.simpleapi import CreateWorkspace

ws_tof = mtd["tof_egs2_1.plot_mcstas_data"]  # your monitor
L1 = 146  # meters – put your real source→monitor distance here

# TOF axis (bin edges)
tof = ws_tof.readX(0)  # shape (N+1,)

# convert µs → Å
lambda_axis = 3.956034 * tof / (1000.0 * L1)

# copy Y and E
y = ws_tof.readY(0)
e = ws_tof.readE(0)

ws_lambda = CreateWorkspace(
    DataX=lambda_axis,
    DataY=y,
    DataE=e,
    NSpec=1,
    UnitX="Wavelength"
)


In [ ]:
# Normalization
l_monitor_wavelength = ws_lambda.readX(0)
l_monitor_intensity = ws_lambda.readY(0)
func_spectra = scipy.interpolate.interp1d(l_monitor_wavelength, l_monitor_intensity)

In [ ]:
l_monitor_wavelength

In [ ]:
np_lambda = numpy.linspace(2., 3.75, num = 25, endpoint=True)
plt.plot(np_lambda, func_spectra(np_lambda))

In [ ]:
def write_to_file(peaks_workspace_name: str, file_out:str):
    peaks_workspace = mtd[peaks_workspace_name]
    np_h = numpy.array(peaks_workspace.column('h'), dtype=float)
    np_k = numpy.array(peaks_workspace.column('k'), dtype=float)
    np_l = numpy.array(peaks_workspace.column('l'), dtype=float)
    np_int = numpy.array(peaks_workspace.column("Intens"), dtype=float)
    np_sint = numpy.array(peaks_workspace.column("SigInt"), dtype=float)
    np_wavelength = numpy.array(peaks_workspace.column("Wavelength"), dtype=float)
    np_d = numpy.array(peaks_workspace.column("DSpacing"), dtype=float)

    np_tth = 2 * numpy.asin(0.5 * np_wavelength / np_d)
    
    # f_name_hkl = "C60_tetra.hkl"
    f_name_hkl = "peaks_scipp.hkl"
    np_hkl_model, np_fsq_model = load_fsq(f_name_hkl)

    np_hkl_order = numpy.square(np_h) + numpy.square(np_k) + numpy.square(np_l)
    np_arg = numpy.argsort(np_hkl_order) 
    ls_out = ["""loop_
_diffrn_refln_index_h
_diffrn_refln_index_k
_diffrn_refln_index_l
_diffrn_refln_intensity
_diffrn_refln_intensity_sigma
_diffrn_refln_fsq_model
_diffrn_refln_intensity_no_norm
_diffrn_refln_intensity_sigma_no_norm
_diffrn_refln_intensity_incident
_diffrn_refln_wavelength
_diffrn_refln_ttheta
_diffrn_refln_d"""]
    flag_normalize_per_wavelength = True
    flag_normalize_per_sin_sq_th = True
    for i in np_arg:
        h, k, l = int(numpy.round(np_h[i], 0)), int(numpy.round(np_k[i], 0)), int(numpy.round(np_l[i], 0))
        flag_model = numpy.logical_and(np_hkl_model[0] == h, numpy.logical_and(np_hkl_model[1] == k, np_hkl_model[2] == l))
        try:
            fsq_model = np_fsq_model[flag_model][0]
        except:
            fsq_model = 9999
        if numpy.isclose(numpy.square(h)+numpy.square(k)+numpy.square(l), 0):
            continue
        iint, siint = np_int[i], np_sint[i]
        wavelength = np_wavelength[i]
        tth = numpy.degrees(np_tth[i])
        d = np_d[i]
        incident = func_spectra(wavelength)
        if wavelength < 2.25 or wavelength > 3.5 :
            continue
        iint_n = iint *  1e8/incident
        siint_n = siint * 1e8/incident

        if flag_normalize_per_wavelength:
            iint_n = iint_n / numpy.power(wavelength, 4)
            siint_n = siint_n / numpy.power(wavelength, 4)
        if flag_normalize_per_sin_sq_th:
            iint_n = 1 * iint_n * (numpy.sin(0.5*numpy.radians(tth)))**2
            siint_n = 1 * siint_n * (numpy.sin(0.5*numpy.radians(tth)))**2
        s_out = f"{h:5.0f} {k:5.0f} {l:5.0f} {iint_n:9.0f} {siint_n:9.0f} {fsq_model:9.0f} {iint:9.2f} {siint:9.2f} {incident:9.0f} {wavelength:7.5f} {tth:7.2f} {d:7.5f}" 
        ls_out.append(s_out)
    with open(file_out,'w') as fid:
        fid.write("\n".join(ls_out))
    return

In [ ]:

def load_fsq(f_name:str):
    with open(f_name, 'r') as fid:
        l_content = fid.readlines()
    l_content = [hh for hh in l_content if not hh.startswith("#")]
    l_hkl, l_fsq = [], []
    for line in l_content:
        l_hh = line.strip().split()
        l_hkl.append((int(l_hh[0]), int(l_hh[1]), int(l_hh[2])))
        # l_fsq.append(l_hh[-1])
        l_fsq.append(l_hh[3])
    np_hkl = numpy.array(l_hkl, dtype=int).transpose()
    np_fsq = numpy.array(l_fsq, dtype = float)
    return np_hkl, np_fsq


In [ ]:
write_to_file("integrated_strong_peaks", 'peaks.hkl')

In [ ]:

# %%
hkl = -3, 3, 3
flag_hkl = numpy.logical_and(np_hkl[0] == hkl[0],
numpy.logical_and(np_hkl[1] == hkl[1], np_hkl[2] == hkl[2]))
np_fsq[flag_hkl]
# %%

# %%

f_name_hkl = "C60_tetra.hkl"
np_hkl, np_fsq = load_fsq(f_name_hkl)
# %%

print(print_table("integrated_strong_peaks"))

# %%

print(print_table("strong_peaks"))


# %%
mtd['strong_peaks'].columnCount()

mtd['strong_peaks'].getTitle()
mtd['strong_peaks'].getNumberPeaks()
mtd['strong_peaks'].getName()
mtd['strong_peaks'].getColumnNames()
mtd['strong_peaks'].column('DSpacing')
mtd['strong_peaks'].column('TOF')
mtd['strong_peaks'].column('Wavelength')
# %%



FindUBUsingFFT(PeaksWorkspace='strong_peaks',MinD='0.5',MaxD='16')
print(mtd["strong_peaks"].sample().hasOrientedLattice())
print(mtd["strong_peaks"].sample().getOrientedLattice().getUB())# %%
# # %% Find UB-matrix by FFT
# tol = 0.2
# FindUBUsingFFT(PeaksWorkspace="strong_peaks", MinD=1, MaxD=10, Tolerance=tol)
# print(mtd["strong_peaks"].sample().hasOrientedLattice())
# print(mtd["strong_peaks"].sample().getOrientedLattice().getUB())# %%
# %% Index peaks by found UB-matrix

# PeakRadius=0.12, BackgroundOuterRadius=0.2, BackgroundInnerRadius=0.16,)



# %%
 


# %%
mtd['integrated_strong_peaks'].getColumnNames()

# %%
mtd['integrated_strong_peaks'].column('Intens') = 0.01*mtd['integrated_strong_peaks'].column('Intens')
# %%
mtd['integrated_strong_peaks'].column('Intens') = 0.01*mtd['integrated_strong_peaks'].column('Intens')

# %%
mtd['integrated_strong_peaks'].column('SigInt')
# %%
dir(mtd['integrated_strong_peaks'])

# %%

SaveHKL(InputWorkspace="integrated_strong_peaks", Filename="peaks.hkl")
# %%
CopySample(InputWorkspace="integrated_strong_peaks", OutputWorkspace="strong_peaks")
# %%

ConvertToDiffractionMDWorkspace(
    InputWorkspace='EventData_mcstas_data', 
    OutputWorkspace='md_space',
    OutputDimensions="HKL")


# %%
mtd["strong_peaks"]
# %%
mtd["sample"]

# %%
help(FindUBUsingLatticeParameters)
# %%


# %%
mtd["integrated_strong_peaks"]
# %%

print(mtd.getObjectNames())
# %%



# %%

# %%

# %%

mtd["mcstas_data"]
# %% Look on the data

mtd["EventData_mcstas_data"]
# %% Look on the data
mtd["EventData_mcstas_data"].getNumberEvents()
# %% 
mtd["EventData_mcstas_data"].getTofMax()
# %% 
mtd["EventData_mcstas_data"].getTofMin()
# %% 
plotSpectrum("EventData_mcstas_data",[0,])
# %% 
mtd["EventData_mcstas_data"].getEventList(39999)
# %% 

mtd["EventData_mcstas_data"].getNumberHistograms()
# %% 
evList = mtd["EventData_mcstas_data"].getSpectrum(39999)
# %% 
Rebin("EventData_mcstas_data", "BinnedData")
# %% Get some basic information
print("Number of events in event List 0: {}".format(evList.getNumberEvents()))
print("Minimum time of flight in event List 0: {}".format(evList.getTofMax()))
print("Maximum time of flight in event List 0: {}".format(evList.getTofMax()))
print("Memory used: {}".format(evList.getMemorySize()))
print("Type of Events: {}".format(evList.getEventType()))

# %% 
mtd["EventData_mcstas_data"]